# ProjeCT360: Leakage-Safe Colab Training and Evaluation

This notebook trains the two model tasks used by the existing Streamlit app:

1. **Volume forecasting:** expected nonnegative reported-incident count for a city and future date.
2. **Offense classification:** broad and specific offense probability distributions from context known before an incident.

The police staffing tab is descriptive. Staffing variables are tested through ablation and are retained as model inputs only when validation performance supports them.

## Evaluation contract

- Model development uses chronological training and validation periods plus rolling-origin folds.
- Rare-class decisions, vocabularies, and lookup tables are fitted without final-test rows.
- Calibration uses validation data only.
- The final test cell is deliberately gated and runs once after model selection.
- Empty outputs in this committed notebook are intentional. Metrics must be generated in Colab and are never fabricated.
- LightGBM/CatBoost boosting rounds are iterations, not neural-network epochs.


## 0. Colab setup

Run this cell once in a fresh Colab runtime. Restart the runtime after it finishes, then continue from the import cell. The numerical stack follows Colab's constraints; estimator versions match the Streamlit deployment environment.


In [ ]:
%pip install -q \
    pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 lightgbm==4.6.0 \
    catboost==1.2.8 xgboost==2.1.4 optuna==4.2.1 imbalanced-learn==0.13.0 \
    joblib==1.5.3 holidays==0.92 matplotlib==3.10.0 \
    'huggingface_hub>=1.5.0,<2.0'
print("Training dependencies installed. Restart the Colab runtime now.")


In [ ]:
import gc
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

GPU_AVAILABLE = shutil.which("nvidia-smi") is not None
print("Python:", sys.version.split()[0])
print("GPU detected:", GPU_AVAILABLE)
if GPU_AVAILABLE:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)


## 1. Configuration

Development mode uses fewer trials, fewer cities, and a shorter history so the complete workflow can be smoke-tested quickly. Full mode uses all valid rows. Keep `RUN_FINAL_TEST=False` until all choices are frozen from validation results.


In [ ]:
# Required project controls
USE_GOOGLE_DRIVE = True
DATA_PATH = Path("/content/drive/MyDrive/CRIME/Datasets/CT_Combined/CT_2016_2024_combined.csv")
PROJECT_ROOT = Path("/content/drive/MyDrive/CRIME/AI4ALL-Crime-Prediction")
OUTPUT_DIR = Path("/content/drive/MyDrive/CRIME/Project360_v2")
RANDOM_STATE = 42
USE_GPU = GPU_AVAILABLE
RUN_FULL_TUNING = False
RUN_PER_CITY_EXPERIMENTS = False
UPLOAD_TO_HUGGING_FACE = False
HF_REPO_ID = "NoVaxiion/project360-assets"
HF_REPO_TYPE = "dataset"
FINAL_TEST_START_DATE = "2024-11-20"

# Safety, speed, and restart controls
DEVELOPMENT_MODE = not RUN_FULL_TUNING
RUN_FINAL_TEST = False
RESUME_FROM_CHECKPOINTS = True
VALIDATION_DAYS = 90
RARE_MIN_COUNT = 500
N_JOBS = 2 if DEVELOPMENT_MODE else 4
OPTUNA_TRIALS = 8 if DEVELOPMENT_MODE else 50
ROLLING_FOLDS = 2 if DEVELOPMENT_MODE else 4
FORECAST_HORIZON = 30
FORECAST_ORIGIN_STRIDE = 30
DEV_CITY_LIMIT = 8
MAX_DEPLOYMENT_MODEL_MB = 120.0
MIN_HIERARCHY_MACRO_F1_GAIN = 0.005
MODEL_VERSION = "2.2.0"

RUN_ID = f"{MODEL_VERSION}_{'dev' if DEVELOPMENT_MODE else 'full'}_{FINAL_TEST_START_DATE}_{RANDOM_STATE}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints" / RUN_ID
PLOTS_DIR = OUTPUT_DIR / "plots"
REPORTS_DIR = OUTPUT_DIR / "reports"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
print("Mode:", "DEVELOPMENT" if DEVELOPMENT_MODE else "FULL")
print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Training data not found: {DATA_PATH}")
required_source_files = [
    PROJECT_ROOT / "feature_engineering.py", PROJECT_ROOT / "training_pipeline.py",
    PROJECT_ROOT / "gc_train_model.ipynb",
]
if not all(path.exists() for path in required_source_files):
    raise FileNotFoundError(
        "PROJECT_ROOT must contain this notebook, feature_engineering.py, and training_pipeline.py. "
        "Clone the GitHub repository into Drive or update PROJECT_ROOT."
    )
data_signature = f"{DATA_PATH.stat().st_size}_{int(DATA_PATH.stat().st_mtime)}"
code_hash = hashlib.sha256(b"".join(path.read_bytes() for path in required_source_files)).hexdigest()[:12]
CHECKPOINT_DIR = CHECKPOINT_DIR / f"{data_signature}_{code_hash}"
for directory in [CHECKPOINT_DIR, PLOTS_DIR, REPORTS_DIR, ARTIFACTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

from feature_engineering import (
    CLASSIFICATION_FEATURES,
    CATEGORICAL_MODEL_FEATURES,
    FORECAST_FEATURES,
    FORECAST_PREDICTION_MODES,
    RESOURCE_FEATURES,
    SCHEMA_VERSION,
    apply_forecast_strategy,
    apply_rare_class_mapping,
    assert_safe_features,
    build_classification_features,
    build_daily_city_panel,
    build_forecast_features,
    fit_classification_artifacts,
    fit_rare_class_mapping,
    infer_location_type,
    map_broad_category,
    make_temporal_boundaries,
    rolling_origin_boundaries,
    temporal_masks,
)
from training_pipeline import (
    artifact_size_mb,
    checkpoint,
    classification_baselines,
    classification_metrics,
    comparison_row,
    forecast_baseline_predictions,
    make_logistic_classifier,
    make_poisson_regression,
    make_xgboost_classifier,
    make_xgboost_count_regressor,
    paired_accuracy_bootstrap,
    paired_absolute_error_bootstrap,
    per_class_metrics,
    predict_timed,
    regression_metrics,
    recursive_forecast_backtest,
    RoutedClassifier,
    select_classification_candidate,
    select_forecast_candidate,
    select_forecast_blend_weight,
    write_manifest,
)

assert_safe_features(FORECAST_FEATURES)
assert_safe_features(CLASSIFICATION_FEATURES)
print("Shared feature schema imported:", SCHEMA_VERSION)


## 2. Leakage audit and feature decisions

| Existing feature/practice | Finding | Decision |
|---|---|---|
| `offense_group`, offense name/category | Reveals the classification target | Remove from inputs; retain only to define labels |
| Rare classes chosen from final 90 days | Final-test contamination | Fit mapping on training labels only |
| `city_hour_common_crime`, `hour_typical_crime` | Target-derived mode included validation/test events | Remove from model inputs |
| Same-day `crime_diversity` | Uses outcomes from the day/hour being predicted | Remove |
| Full-data location totals/frequencies | Future-data leakage | Rebuild and freeze from training rows only |
| Incident-row category lags | A row shift is not a calendar-day lag and exposes nearby event labels | Remove from classifier |
| Forecast lags/rolling values | Valid only on a complete city/date panel with current row excluded | Rebuilt in shared module |
| Arbitrary category codes + ordinary SMOTE | Synthetic interpolation between nominal codes is invalid | Native categorical models; SMOTENC only in controlled ablation |
| SMOTE plus class and sample weights | Duplicate imbalance correction | Compare one strategy at a time |
| Test set used for early stopping/calibration/model choice | Test contamination | Validation subperiods only; final test locked |
| Independent L1/L2 models called hierarchical | L1 did not route L2 | Describe as broad and specific models; compare true routing separately |
| Staffing/crime-rate variables | Potential reporting/enforcement proxies | Retain only if validation ablation improves trustworthy metrics |
| Per-city models | Very large and previously worse than global fallback | Disabled by default; retain only after repeated validation wins |


## 3. Memory-conscious data loading

Only columns required for training and descriptive app data are read. Chunks are typed before concatenation. Development mode then limits cities and history without mixing future rows into earlier periods.


In [ ]:
USE_COLUMNS = [
    "year", "month", "day", "hour", "city", "location_area", "offense_category_name",
    "population", "total_officers", "male_officer", "female_officer",
    "officers_per_1000_people", "crime_rate_per_1000_people",
]
DTYPES = {
    "year": "int16", "month": "int8", "day": "int8", "hour": "float32",
    "city": "string", "location_area": "string", "offense_category_name": "string",
    "population": "float32", "total_officers": "float32", "male_officer": "float32",
    "female_officer": "float32", "officers_per_1000_people": "float32",
    "crime_rate_per_1000_people": "float32",
}

def load_training_data(path, chunksize=150_000):
    print("Loading selected columns in chunks...")
    chunks = []
    for index, chunk in enumerate(pd.read_csv(path, usecols=lambda c: c.lower().strip() in USE_COLUMNS, chunksize=chunksize)):
        chunk.columns = chunk.columns.str.lower().str.strip()
        for column, dtype in DTYPES.items():
            if column in chunk:
                chunk[column] = chunk[column].astype(dtype)
        chunks.append(chunk)
        print(f"  chunk {index + 1}: {len(chunk):,} rows")
    data = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    data["date"] = pd.to_datetime(data[["year", "month", "day"]], errors="coerce")
    data["hour"] = data["hour"].fillna(0).clip(0, 23).astype("int8")
    data = data.dropna(subset=["date", "city", "location_area", "offense_category_name"])
    for column in ["city", "location_area", "offense_category_name"]:
        data[column] = data[column].astype("category")
    return data

typed_data_checkpoint = CHECKPOINT_DIR / "typed_training_data.pkl"
if RESUME_FROM_CHECKPOINTS and typed_data_checkpoint.exists():
    print("Loading typed data checkpoint:", typed_data_checkpoint)
    df = joblib.load(typed_data_checkpoint)
else:
    df = load_training_data(DATA_PATH)
    if DEVELOPMENT_MODE:
        largest_cities = df["city"].value_counts().head(DEV_CITY_LIMIT).index
        dev_start = max(df["date"].min(), pd.Timestamp(FINAL_TEST_START_DATE) - pd.DateOffset(years=3))
        df = df[df["city"].isin(largest_cities) & (df["date"] >= dev_start)].copy()
        df["city"] = df["city"].cat.remove_unused_categories()
    joblib.dump(df, typed_data_checkpoint, compress=3)

print(f"Rows: {len(df):,}")
print(f"Date range: {df['date'].min().date()} through {df['date'].max().date()}")
print(f"Cities: {df['city'].nunique()}; specific labels: {df['offense_category_name'].nunique()}")


## 4. Immutable chronological periods

The final test period is never passed to tuning, early stopping, rare-label decisions, preprocessing, calibration, or model selection. Validation is split internally into tuning, calibration, and scoring slices.


In [ ]:
boundaries = make_temporal_boundaries(
    df["date"], final_test_start=FINAL_TEST_START_DATE, validation_days=VALIDATION_DAYS
)
masks = temporal_masks(df["date"], boundaries)
assert masks["train"].any() and masks["validation"].any() and masks["test"].any()

validation_dates = sorted(df.loc[masks["validation"], "date"].dt.normalize().unique())
first_cut = validation_dates[len(validation_dates) // 3]
second_cut = validation_dates[2 * len(validation_dates) // 3]
validation_tune_mask = masks["validation"] & (df["date"] < first_cut)
calibration_mask = masks["validation"] & df["date"].between(first_cut, second_cut - pd.Timedelta(days=1))
validation_score_mask = masks["validation"] & (df["date"] >= second_cut)

periods = {
    "training_period": f"{df.loc[masks['train'], 'date'].min().date()} to {boundaries.train_end.date()}",
    "validation_period": f"{boundaries.validation_start.date()} to {boundaries.validation_end.date()}",
    "test_period": f"{boundaries.test_start.date()} to {boundaries.test_end.date()}",
}
(REPORTS_DIR / "temporal_boundaries.json").write_text(json.dumps({**boundaries.to_dict(), **periods}, indent=2))
print(json.dumps(boundaries.to_dict(), indent=2))

rare_mapping = fit_rare_class_mapping(
    df.loc[masks["train"], "offense_category_name"].astype(str), RARE_MIN_COUNT
)
df["offense_category_clean"] = apply_rare_class_mapping(
    df["offense_category_name"].astype(str), rare_mapping
).to_numpy()
assert set(rare_mapping["rare_labels"]) == set(
    pd.Series(df.loc[masks["train"], "offense_category_name"].astype(str)).value_counts()
    .loc[lambda counts: counts < RARE_MIN_COUNT].index
)
_ = (REPORTS_DIR / "rare_class_mapping.json").write_text(json.dumps(rare_mapping, indent=2))


## 5. Volume features, baselines, and walk-forward folds

The daily panel explicitly inserts missing city/date combinations as zero counts. Therefore lag 1 means the previous calendar day, not the previous incident row. Every rolling feature uses `shift(1)`. The one-step table is diagnostic; selection uses app-aligned recursive 30-day origins where holdout outcomes are never fed back into later horizons.


In [ ]:
daily_checkpoint = CHECKPOINT_DIR / "daily_city_panel.pkl"
daily = checkpoint(daily_checkpoint, lambda: build_daily_city_panel(df), RESUME_FROM_CHECKPOINTS)
forecast_feature_checkpoint = CHECKPOINT_DIR / "forecast_feature_frame.pkl"
if RESUME_FROM_CHECKPOINTS and forecast_feature_checkpoint.exists():
    forecast_frame, forecast_artifacts = joblib.load(forecast_feature_checkpoint)
else:
    forecast_frame, forecast_artifacts = build_forecast_features(daily)
    forecast_frame = forecast_frame.dropna(subset=FORECAST_FEATURES + ["crime_count"]).reset_index(drop=True)
    joblib.dump((forecast_frame, forecast_artifacts), forecast_feature_checkpoint, compress=3)
forecast_masks = temporal_masks(forecast_frame["date"], boundaries)

X_train_f = forecast_frame.loc[forecast_masks["train"], FORECAST_FEATURES]
y_train_f = forecast_frame.loc[forecast_masks["train"], "crime_count"].to_numpy()
X_valid_f = forecast_frame.loc[forecast_masks["validation"], FORECAST_FEATURES]
y_valid_f = forecast_frame.loc[forecast_masks["validation"], "crime_count"].to_numpy()

assert forecast_frame.loc[forecast_masks["train"], "date"].max() < forecast_frame.loc[forecast_masks["validation"], "date"].min()
assert forecast_frame.loc[forecast_masks["validation"], "date"].max() < forecast_frame.loc[forecast_masks["test"], "date"].min()

baseline_rows = []
baseline_predictions = forecast_baseline_predictions(
    forecast_frame.loc[forecast_masks["validation"]], forecast_frame.loc[forecast_masks["train"]]
)
for name, prediction in baseline_predictions.items():
    baseline_rows.append({"model_name": name, **regression_metrics(y_valid_f, prediction), "kind": "baseline"})
forecast_one_step_baseline_table = pd.DataFrame(baseline_rows).sort_values("mae")
print("One-step diagnostic baselines (not used for deployment selection):")
display(forecast_one_step_baseline_table)

recursive_validation_baseline = recursive_forecast_backtest(
    None, daily, forecast_artifacts, boundaries.validation_start, boundaries.validation_end,
    horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE, model_weight=0.0,
)
recursive_baseline_metrics = regression_metrics(
    recursive_validation_baseline["actual"], recursive_validation_baseline["predicted"]
)
forecast_baseline_table = pd.DataFrame([{
    "model_name": "Recursive seven-day rolling average",
    **recursive_baseline_metrics, "kind": "30-day baseline",
}])
print("Leakage-safe recursive 30-day deployment baseline:")
display(forecast_baseline_table)

walk_forward_folds = rolling_origin_boundaries(
    boundaries.train_end, forecast_frame["date"].min(), validation_days=FORECAST_HORIZON, folds=ROLLING_FOLDS
)
print("Walk-forward folds:")
for fold in walk_forward_folds:
    print(tuple(str(value.date()) for value in fold))


## 6. Controlled forecast model comparison and tuning

Candidate selection uses recursive 30-day validation MAE first, with RMSE/WAPE/bias and artifact size as supporting criteria. The final test remains untouched. Optuna tunes LightGBM objectives on pre-test 30-day walk-forward folds and checkpoints its study in SQLite. L1 candidates directly optimize MAE, while the residual candidate learns a correction to the seven-day baseline.


In [ ]:
import lightgbm as lgb
import optuna
from catboost import CatBoostRegressor

def tune_forecaster():
    storage = f"sqlite:///{CHECKPOINT_DIR / 'forecast_optuna.db'}"
    study = optuna.create_study(
        study_name="project360_forecast_v2", direction="minimize", storage=storage, load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE), pruner=optuna.pruners.MedianPruner(),
    )
    def objective(trial):
        objective_name = trial.suggest_categorical("objective", ["poisson", "tweedie", "regression_l1"])
        params = {
            "objective": objective_name,
            "n_estimators": 1200,
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 20, 96),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 120),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 4.0),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "random_state": RANDOM_STATE, "n_jobs": N_JOBS, "verbosity": -1,
        }
        if objective_name == "tweedie":
            params["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", 1.1, 1.8)
        scores = []
        for fold_index, (fold_train_end, fold_start, fold_end) in enumerate(walk_forward_folds):
            train_fold = forecast_frame["date"] <= fold_train_end
            valid_fold = forecast_frame["date"].between(fold_start, fold_end)
            model = lgb.LGBMRegressor(**params)
            model.fit(
                forecast_frame.loc[train_fold, FORECAST_FEATURES], forecast_frame.loc[train_fold, "crime_count"],
                categorical_feature=["city_code"],
                eval_set=[(forecast_frame.loc[valid_fold, FORECAST_FEATURES], forecast_frame.loc[valid_fold, "crime_count"])],
                callbacks=[lgb.early_stopping(80, verbose=False)],
            )
            fold_backtest = recursive_forecast_backtest(
                model, daily, forecast_artifacts, fold_start, fold_end,
                horizon=FORECAST_HORIZON, stride=FORECAST_HORIZON,
                prediction_mode="direct", model_weight=1.0,
            )
            scores.append(regression_metrics(fold_backtest["actual"], fold_backtest["predicted"])["mae"])
            trial.report(np.mean(scores), fold_index)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, gc_after_trial=True)
    return study.best_params

best_lgb_params = tune_forecaster()
print("Best validation-only LightGBM parameters:", best_lgb_params)


In [ ]:
def fit_forecast_candidate(name):
    evaluation_history = {}
    prediction_mode = "residual_to_roll_mean_7" if name == "LightGBM Residual L1" else "direct"
    fit_y_train = y_train_f - X_train_f["roll_mean_7"].to_numpy() if prediction_mode.startswith("residual") else y_train_f
    fit_y_valid = y_valid_f - X_valid_f["roll_mean_7"].to_numpy() if prediction_mode.startswith("residual") else y_valid_f
    tuned_lgb = {key: value for key, value in best_lgb_params.items() if key not in {"objective", "tweedie_variance_power"}}
    common_lgb = dict(
        n_estimators=1200, learning_rate=0.03, num_leaves=48, min_child_samples=40,
        reg_alpha=0.2, reg_lambda=1.0, subsample=0.85, subsample_freq=1, colsample_bytree=0.85,
        random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=-1,
    )
    if name in {"LightGBM Poisson", "LightGBM Tweedie", "LightGBM L1", "LightGBM Residual L1"}:
        objective_name = {
            "LightGBM Poisson": "poisson", "LightGBM Tweedie": "tweedie",
            "LightGBM L1": "regression_l1", "LightGBM Residual L1": "regression_l1",
        }[name]
        model_params = {**common_lgb, **tuned_lgb, "objective": objective_name}
        if objective_name == "tweedie":
            model_params["tweedie_variance_power"] = best_lgb_params.get("tweedie_variance_power", 1.3)
        model = lgb.LGBMRegressor(**model_params)
        eval_metric = "poisson" if objective_name == "poisson" else "l1" if objective_name == "regression_l1" else "rmse"
        fit_kwargs = {"categorical_feature": ["city_code"], "eval_set": [(X_valid_f, fit_y_valid)], "eval_names": ["Validation"], "eval_metric": eval_metric, "callbacks": [lgb.early_stopping(100, verbose=False), lgb.record_evaluation(evaluation_history)]}
    elif name == "Regularized Poisson":
        model = make_poisson_regression(FORECAST_FEATURES)
        fit_kwargs = {}
    elif name in {"CatBoost Poisson", "CatBoost MAE"}:
        loss_name = "Poisson" if name == "CatBoost Poisson" else "MAE"
        model = CatBoostRegressor(
            loss_function=loss_name, eval_metric=loss_name, iterations=900, depth=8, learning_rate=0.04,
            random_seed=RANDOM_STATE, verbose=False, task_type="GPU" if USE_GPU and name == "CatBoost Poisson" else "CPU",
        )
        fit_kwargs = {"cat_features": [FORECAST_FEATURES.index("city_code")], "eval_set": (X_valid_f, fit_y_valid), "early_stopping_rounds": 80}
    elif name == "XGBoost Count":
        model = make_xgboost_count_regressor(FORECAST_FEATURES, RANDOM_STATE, N_JOBS)
        fit_kwargs = {}
    else:
        raise KeyError(name)
    started = time.perf_counter()
    model.fit(X_train_f, fit_y_train, **fit_kwargs)
    training_seconds = time.perf_counter() - started
    raw_prediction, inference_ms = predict_timed(model, X_valid_f)
    one_step_prediction = apply_forecast_strategy(
        raw_prediction, X_valid_f, prediction_mode=prediction_mode, model_weight=1.0
    )
    size = artifact_size_mb(model, CHECKPOINT_DIR / f"forecast_{name.lower().replace(' ', '_')}.pkl")
    return {"model": model, "prediction_mode": prediction_mode,
            "one_step_prediction": one_step_prediction, "one_step_metrics": regression_metrics(y_valid_f, one_step_prediction),
            "training_seconds": training_seconds, "inference_ms": inference_ms, "artifact_size_mb": size,
            "evaluation_history": evaluation_history}

forecast_candidates = {}
for candidate_name in ["LightGBM Poisson", "LightGBM Tweedie", "LightGBM L1", "LightGBM Residual L1", "CatBoost Poisson", "CatBoost MAE", "XGBoost Count", "Regularized Poisson"]:
    print("Training", candidate_name)
    forecast_candidates[candidate_name] = checkpoint(
        CHECKPOINT_DIR / f"result_forecast_{candidate_name.lower().replace(' ', '_')}.pkl",
        lambda name=candidate_name: fit_forecast_candidate(name),
        RESUME_FROM_CHECKPOINTS,
    )

forecast_candidate_backtests = {}
forecast_validation_rows = []
for name, result in forecast_candidates.items():
    backtest = recursive_forecast_backtest(
        result["model"], daily, forecast_artifacts, boundaries.validation_start, boundaries.validation_end,
        horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE,
        prediction_mode=result["prediction_mode"], model_weight=1.0,
    )
    forecast_candidate_backtests[name] = backtest
    recursive_metrics = regression_metrics(backtest["actual"], backtest["predicted"])
    result["metrics"] = recursive_metrics
    forecast_validation_rows.append({
        "model_name": name, **recursive_metrics, "one_step_mae": result["one_step_metrics"]["mae"],
        "prediction_mode": result["prediction_mode"], "artifact_size_mb": result["artifact_size_mb"],
        "training_time_seconds": result["training_seconds"], "inference_ms_per_row": result["inference_ms"],
    })
forecast_validation = pd.DataFrame(forecast_validation_rows)
selected_forecast_name = select_forecast_candidate(forecast_validation)
selected_forecast_candidate_name = selected_forecast_name
selected_forecaster = forecast_candidates[selected_forecast_name]["model"]
selected_forecast_prediction_mode = forecast_candidates[selected_forecast_name]["prediction_mode"]
selected_forecast_validation_rows = forecast_candidate_backtests[selected_forecast_name]
selected_forecast_validation_metrics = regression_metrics(
    selected_forecast_validation_rows["actual"], selected_forecast_validation_rows["predicted"]
)
display(forecast_validation.sort_values("mae"))
print("Selected from recursive 30-day validation only:", selected_forecast_name)


## 7. Resource-feature ablation and optional per-city forecast experiment

Staffing and crime-rate variables are potentially biased proxies. This cell compares the selected model family with and without those inputs, then learns a constrained model-versus-rolling-baseline weight from recursive validation trajectories. It never keeps resource features merely because they are available. Per-city models remain diagnostic and are never exported by this workflow.


In [ ]:
from sklearn.base import clone

def fit_ablation_features(features):
    candidate_name = selected_forecast_candidate_name
    candidate_result = forecast_candidates[candidate_name]
    prediction_mode = candidate_result["prediction_mode"]
    fit_y_train = y_train_f - X_train_f["roll_mean_7"].to_numpy() if prediction_mode.startswith("residual") else y_train_f
    fit_y_valid = y_valid_f - X_valid_f["roll_mean_7"].to_numpy() if prediction_mode.startswith("residual") else y_valid_f
    if candidate_name == "Regularized Poisson":
        model = make_poisson_regression(features)
    elif candidate_name == "XGBoost Count":
        model = make_xgboost_count_regressor(features, RANDOM_STATE, N_JOBS)
    else:
        model = clone(candidate_result["model"])
    fit_kwargs = {}
    if candidate_name.startswith("LightGBM"):
        objective_name = model.get_params().get("objective")
        eval_metric = "poisson" if objective_name == "poisson" else "l1" if objective_name == "regression_l1" else "rmse"
        fit_kwargs = {
            "categorical_feature": ["city_code"],
            "eval_set": [(forecast_frame.loc[forecast_masks["validation"], features], fit_y_valid)],
            "eval_metric": eval_metric,
            "callbacks": [lgb.early_stopping(100, verbose=False)],
        }
    elif candidate_name.startswith("CatBoost"):
        fit_kwargs = {
            "cat_features": [features.index("city_code")],
            "eval_set": (forecast_frame.loc[forecast_masks["validation"], features], fit_y_valid),
            "early_stopping_rounds": 80,
        }
    model.fit(forecast_frame.loc[forecast_masks["train"], features], fit_y_train, **fit_kwargs)
    return model

forecast_without_resources = [column for column in FORECAST_FEATURES if column not in RESOURCE_FEATURES]
ablation_model = fit_ablation_features(forecast_without_resources)
full_artifacts = {**forecast_artifacts, "feature_columns": list(FORECAST_FEATURES)}
ablation_artifacts = {**forecast_artifacts, "feature_columns": forecast_without_resources}
full_weight, full_blend_table, full_backtest = select_forecast_blend_weight(
    selected_forecaster, daily, full_artifacts, boundaries.validation_start, boundaries.validation_end,
    prediction_mode=selected_forecast_prediction_mode, horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE,
)
ablation_weight, ablation_blend_table, ablation_backtest = select_forecast_blend_weight(
    ablation_model, daily, ablation_artifacts, boundaries.validation_start, boundaries.validation_end,
    prediction_mode=selected_forecast_prediction_mode, horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE,
)
full_metrics = regression_metrics(full_backtest["actual"], full_backtest["predicted"])
ablation_metrics = regression_metrics(ablation_backtest["actual"], ablation_backtest["predicted"])
blend_comparison = pd.concat([
    full_blend_table.assign(feature_variant="with resource proxies"),
    ablation_blend_table.assign(feature_variant="without resource proxies"),
], ignore_index=True)
display(blend_comparison.sort_values(["mae", "feature_variant"]).head(12))
KEEP_FORECAST_RESOURCES = full_metrics["mae"] < ablation_metrics["mae"]
if KEEP_FORECAST_RESOURCES:
    selected_forecast_validation_rows = full_backtest
    selected_forecast_model_weight = full_weight
    forecast_artifacts = full_artifacts
    print("Resource features improved recursive validation MAE; retain with model-card proxy warning.")
else:
    selected_forecaster = ablation_model
    selected_forecast_name = f"{selected_forecast_candidate_name} (without resource proxies)"
    selected_forecast_validation_rows = ablation_backtest
    selected_forecast_model_weight = ablation_weight
    forecast_artifacts = ablation_artifacts
    print("Resource features did not improve recursive validation MAE; exclude them.")
forecast_artifacts.update({
    "prediction_mode": selected_forecast_prediction_mode,
    "model_weight": float(selected_forecast_model_weight),
    "forecast_horizon": FORECAST_HORIZON,
    "validation_origin_stride": FORECAST_ORIGIN_STRIDE,
    "selection_metric": "recursive_30_day_mae",
})
selected_forecast_validation_metrics = regression_metrics(
    selected_forecast_validation_rows["actual"], selected_forecast_validation_rows["predicted"]
)
best_forecast_baseline = forecast_baseline_table.iloc[0]
FORECAST_VALIDATION_PASSED = selected_forecast_validation_metrics["mae"] < float(best_forecast_baseline["mae"])
print(f"Selected model weight: {selected_forecast_model_weight:.2f}")
if FORECAST_VALIDATION_PASSED:
    print(f"Forecast validation passed: MAE {selected_forecast_validation_metrics['mae']:.4f} vs baseline {best_forecast_baseline['mae']:.4f}.")
else:
    print(f"WARNING: forecast validation failed: MAE {selected_forecast_validation_metrics['mae']:.4f} vs baseline {best_forecast_baseline['mae']:.4f}. Classification may continue, but this forecaster cannot be final-tested or exported.")

if RUN_PER_CITY_EXPERIMENTS:
    print("Per-city models are diagnostic only and are not exported by this compact workflow.")
    # Intentionally conservative: add a city only when its validation MAE is lower than the global model.
    for city in forecast_frame.loc[forecast_masks["train"], "city"].value_counts().head(10).index:
        train_city = forecast_masks["train"] & (forecast_frame["city"] == city)
        valid_city = forecast_masks["validation"] & (forecast_frame["city"] == city)
        if train_city.sum() < 730 or valid_city.sum() < 28:
            continue
        city_model = lgb.LGBMRegressor(objective="regression_l1", n_estimators=500, learning_rate=0.04,
                                       num_leaves=24, min_child_samples=30, random_state=RANDOM_STATE,
                                       n_jobs=1, verbosity=-1)
        city_features = [f for f in forecast_artifacts["feature_columns"] if f != "city_code"]
        city_model.fit(forecast_frame.loc[train_city, city_features], forecast_frame.loc[train_city, "crime_count"])
        city_artifacts = {**forecast_artifacts, "feature_columns": city_features}
        city_backtest = recursive_forecast_backtest(
            city_model, daily, city_artifacts, boundaries.validation_start, boundaries.validation_end,
            horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE, cities=[str(city)],
        )
        global_backtest = recursive_forecast_backtest(
            selected_forecaster, daily, forecast_artifacts, boundaries.validation_start, boundaries.validation_end,
            horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE,
            prediction_mode=selected_forecast_prediction_mode, model_weight=selected_forecast_model_weight, cities=[str(city)],
        )
        city_mae = regression_metrics(city_backtest["actual"], city_backtest["predicted"])["mae"]
        global_mae = regression_metrics(global_backtest["actual"], global_backtest["predicted"])["mae"]
        print(f"  {city}: city MAE={city_mae:.3f}; global MAE={global_mae:.3f}")
    print("Per-city results are diagnostic only. Export requires wins across every rolling fold, so no per-city models are retained in this run.")


## 8. Classification features and baselines

The frozen lookups below contain only target-independent counts from the training period. `hour_typical_crime`, same-day diversity, event labels, and full-dataset location counts are intentionally absent.


In [ ]:
from sklearn.preprocessing import LabelEncoder

df["location_type"] = df["location_area"].astype(str).map(infer_location_type)
train_incidents = df.loc[masks["train"]].copy()
classification_feature_checkpoint = CHECKPOINT_DIR / "classification_feature_frame.pkl"
if RESUME_FROM_CHECKPOINTS and classification_feature_checkpoint.exists():
    X_all_c, classification_artifacts = joblib.load(classification_feature_checkpoint)
else:
    classification_artifacts = fit_classification_artifacts(train_incidents, include_resources=True)
    X_all_c = build_classification_features(df, classification_artifacts)
    joblib.dump((X_all_c, classification_artifacts), classification_feature_checkpoint, compress=3)
assert list(X_all_c.columns) == classification_artifacts["feature_columns"]
assert not set(X_all_c.columns) & {"offense_group", "offense_name", "offense_category_name", "offense_category_clean"}

specific_encoder = LabelEncoder().fit(df.loc[masks["train"], "offense_category_clean"].astype(str))
specific_class_to_id = {name: index for index, name in enumerate(specific_encoder.classes_)}
specific_labels = df["offense_category_clean"].astype(str).map(specific_class_to_id)
if specific_labels.isna().any():
    unknown = sorted(df.loc[specific_labels.isna(), "offense_category_clean"].astype(str).unique())
    raise ValueError(f"Final periods contain labels absent from training after rare mapping: {unknown}")
specific_labels = specific_labels.astype(int).to_numpy()

broad_names = df["offense_category_clean"].map(map_broad_category)
broad_encoder = LabelEncoder().fit(broad_names.loc[masks["train"]])
broad_labels = broad_encoder.transform(broad_names)

train_context = df.loc[masks["train"], ["city", "location_type", "hour"]].copy()
validation_context = df.loc[validation_score_mask, ["city", "location_type", "hour"]].copy()
specific_baselines = classification_baselines(
    train_context, specific_labels[masks["train"].to_numpy()], validation_context, len(specific_encoder.classes_)
)
specific_baseline_table = pd.DataFrame([
    {"model_name": name, **classification_metrics(specific_labels[validation_score_mask.to_numpy()], probs)}
    for name, probs in specific_baselines.items()
]).sort_values("macro_f1", ascending=False)
display(specific_baseline_table)


## 9. Imbalance strategy study

Only one correction is applied per experiment: none, class weights, random undersampling, or SMOTENC with nominal categorical columns declared explicitly. The strategy is selected on validation macro-F1/log loss, never final-test accuracy.


In [ ]:
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils.class_weight import compute_sample_weight

X_train_c = X_all_c.loc[masks["train"]]
y_train_c = specific_labels[masks["train"].to_numpy()]
X_tune_c = X_all_c.loc[validation_tune_mask]
y_tune_c = specific_labels[validation_tune_mask.to_numpy()]

def resample_training(strategy):
    if strategy == "none":
        return X_train_c, y_train_c, None
    if strategy == "class_weight":
        return X_train_c, y_train_c, compute_sample_weight("balanced", y_train_c)
    if strategy == "random_undersampling":
        sampler = RandomUnderSampler(random_state=RANDOM_STATE)
        X_resampled, y_resampled = sampler.fit_resample(X_train_c, y_train_c)
        return X_resampled, y_resampled, None
    if strategy == "smotenc":
        categorical_indices = [X_train_c.columns.get_loc(column) for column in CATEGORICAL_MODEL_FEATURES]
        counts = pd.Series(y_train_c).value_counts()
        target_count = int(counts.median())
        sampling_strategy = {int(label): target_count for label, count in counts.items() if count < target_count}
        if not sampling_strategy:
            return X_train_c, y_train_c, None
        sampler = SMOTENC(categorical_features=categorical_indices, sampling_strategy=sampling_strategy,
                          random_state=RANDOM_STATE, k_neighbors=3)
        X_resampled, y_resampled = sampler.fit_resample(X_train_c, y_train_c)
        return pd.DataFrame(X_resampled, columns=X_train_c.columns), y_resampled, None
    raise KeyError(strategy)

imbalance_results = []
imbalance_models = {}
imbalance_histories = {}
for strategy in ["none", "class_weight", "random_undersampling", "smotenc"]:
    print("Testing imbalance strategy:", strategy)
    X_fit, y_fit, sample_weight = resample_training(strategy)
    model = lgb.LGBMClassifier(
        objective="multiclass", n_estimators=800, learning_rate=0.04, num_leaves=48,
        min_child_samples=40, reg_alpha=0.2, reg_lambda=1.5, random_state=RANDOM_STATE,
        n_jobs=N_JOBS, verbosity=-1,
    )
    evaluation_history = {}
    model.fit(X_fit, y_fit, sample_weight=sample_weight, categorical_feature=CATEGORICAL_MODEL_FEATURES,
              eval_set=[(X_tune_c, y_tune_c)], eval_names=["Validation"], eval_metric="multi_logloss",
              callbacks=[lgb.early_stopping(60, verbose=False), lgb.record_evaluation(evaluation_history)])
    probabilities = model.predict_proba(X_tune_c)
    metrics = classification_metrics(y_tune_c, probabilities)
    imbalance_results.append({"strategy": strategy, **metrics})
    imbalance_models[strategy] = model
    imbalance_histories[strategy] = evaluation_history
    del X_fit, y_fit
    gc.collect()

imbalance_table = pd.DataFrame(imbalance_results).sort_values(
    ["macro_f1", "log_loss"], ascending=[False, True]
)
selected_imbalance = imbalance_table.iloc[0]["strategy"]
display(imbalance_table)
print("Selected imbalance method from validation:", selected_imbalance)


## 10. Classification model comparison

Logistic regression is the interpretable baseline. LightGBM, CatBoost, and XGBoost are compared on the same chronological validation scoring slice. Candidate choice prioritizes macro-F1, then log loss/calibration, balanced accuracy, top-3 accuracy, and artifact size.


In [ ]:
from catboost import CatBoostClassifier

X_score_c = X_all_c.loc[validation_score_mask]
y_score_c = specific_labels[validation_score_mask.to_numpy()]

def fit_classifier_candidate(name):
    if name == "Multinomial Logistic Regression":
        model = make_logistic_classifier(CLASSIFICATION_FEATURES, CATEGORICAL_MODEL_FEATURES)
        fit_kwargs = {}
        imbalance_method = "class_weight"
    elif name == "LightGBM":
        model = imbalance_models[selected_imbalance]
        prediction, inference_ms = predict_timed(model, X_score_c, probabilities=True)
        return {"model": model, "probabilities": prediction, "training_seconds": 0.0,
                "inference_ms": inference_ms, "imbalance_method": selected_imbalance,
                "artifact_size_mb": artifact_size_mb(model, CHECKPOINT_DIR / "classifier_lightgbm.pkl")}
    elif name == "CatBoost":
        model = CatBoostClassifier(
            loss_function="MultiClass", iterations=900, depth=8, learning_rate=0.04,
            random_seed=RANDOM_STATE, verbose=False, auto_class_weights="Balanced",
            task_type="GPU" if USE_GPU else "CPU",
        )
        fit_kwargs = {"cat_features": CATEGORICAL_MODEL_FEATURES, "eval_set": (X_tune_c, y_tune_c), "early_stopping_rounds": 60}
        imbalance_method = "class_weight"
    elif name == "XGBoost":
        model = make_xgboost_classifier(
            CLASSIFICATION_FEATURES, CATEGORICAL_MODEL_FEATURES,
            len(specific_encoder.classes_), RANDOM_STATE, N_JOBS,
        )
        fit_kwargs = {}
        imbalance_method = "none"
    else:
        raise KeyError(name)
    started = time.perf_counter()
    model.fit(X_train_c, y_train_c, **fit_kwargs)
    training_seconds = time.perf_counter() - started
    probabilities, inference_ms = predict_timed(model, X_score_c, probabilities=True)
    size = artifact_size_mb(model, CHECKPOINT_DIR / f"classifier_{name.lower().replace(' ', '_')}.pkl")
    return {"model": model, "probabilities": probabilities, "training_seconds": training_seconds,
            "imbalance_method": imbalance_method,
            "inference_ms": inference_ms, "artifact_size_mb": size}

classifier_candidates = {}
for candidate_name in ["Multinomial Logistic Regression", "LightGBM", "CatBoost", "XGBoost"]:
    print("Training", candidate_name)
    classifier_candidates[candidate_name] = checkpoint(
        CHECKPOINT_DIR / f"result_classifier_{candidate_name.lower().replace(' ', '_')}.pkl",
        lambda name=candidate_name: fit_classifier_candidate(name), RESUME_FROM_CHECKPOINTS,
    )

classification_validation = pd.DataFrame([
    {"model_name": name, **classification_metrics(y_score_c, result["probabilities"]),
     "artifact_size_mb": result["artifact_size_mb"], "training_time_seconds": result["training_seconds"],
     "inference_ms_per_row": result["inference_ms"]}
    for name, result in classifier_candidates.items()
])
selected_specific_name = select_classification_candidate(
    classification_validation, max_artifact_mb=MAX_DEPLOYMENT_MODEL_MB
)
selected_specific_candidate_name = selected_specific_name
selected_specific_base = classifier_candidates[selected_specific_name]["model"]
display(classification_validation.sort_values(["macro_f1", "log_loss"], ascending=[False, True]))
print("Selected specific model from validation only:", selected_specific_name)


## 11. Validation-only calibration and broad model

Calibration is fitted on the middle validation slice and compared on the final validation slice. Isotonic is considered only when the calibration slice has enough examples per class. Accuracy is not the calibration selection criterion.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

X_cal_c = X_all_c.loc[calibration_mask]
y_cal_c = specific_labels[calibration_mask.to_numpy()]

def calibration_candidates(base_model, X_cal, y_cal, X_score, y_score):
    candidates = {"uncalibrated": base_model}
    for method in ["sigmoid", "isotonic"]:
        if method == "isotonic" and pd.Series(y_cal).value_counts().min() < 100:
            continue
        calibrated = CalibratedClassifierCV(FrozenEstimator(base_model), method=method)
        calibrated.fit(X_cal, y_cal)
        candidates[method] = calibrated
    rows = []
    for name, model in candidates.items():
        probabilities = model.predict_proba(X_score)
        rows.append({"calibration": name, **classification_metrics(y_score, probabilities)})
    table = pd.DataFrame(rows).sort_values(["log_loss", "ece", "brier_multiclass"])
    return candidates[table.iloc[0]["calibration"]], table.iloc[0]["calibration"], table

selected_specific_model, selected_calibration, calibration_table = calibration_candidates(
    selected_specific_base, X_cal_c, y_cal_c, X_score_c, y_score_c
)
display(calibration_table)
print("Selected calibration:", selected_calibration)

# Broad model uses the same safe feature matrix and validation protocol.
y_train_broad = broad_labels[masks["train"].to_numpy()]
y_tune_broad = broad_labels[validation_tune_mask.to_numpy()]
y_cal_broad = broad_labels[calibration_mask.to_numpy()]
y_score_broad = broad_labels[validation_score_mask.to_numpy()]
broad_base = lgb.LGBMClassifier(
    objective="multiclass", n_estimators=700, learning_rate=0.04, num_leaves=40,
    min_child_samples=50, class_weight="balanced", reg_lambda=1.5,
    random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=-1,
)
broad_base.fit(X_train_c, y_train_broad, categorical_feature=CATEGORICAL_MODEL_FEATURES,
               eval_set=[(X_tune_c, y_tune_broad)], callbacks=[lgb.early_stopping(60, verbose=False)])
selected_broad_model, broad_calibration, broad_calibration_table = calibration_candidates(
    broad_base, X_cal_c, y_cal_broad, X_score_c, y_score_broad
)
display(broad_calibration_table)


## 12. Classification resource ablation and true hierarchy check

The ablation removes population, staffing, and crime-rate proxies together. The simpler feature set wins unless the full set improves validation macro-F1 without worsening log loss. A true routed hierarchy is evaluated separately; independent broad/specific predictions are not mislabeled as hierarchical.


In [ ]:
artifacts_without_resources = fit_classification_artifacts(train_incidents, include_resources=False)
X_no_resources = build_classification_features(df, artifacts_without_resources)
resource_ablation_model = lgb.LGBMClassifier(
    objective="multiclass", n_estimators=700, learning_rate=0.04, num_leaves=48,
    min_child_samples=40, class_weight="balanced", random_state=RANDOM_STATE,
    n_jobs=N_JOBS, verbosity=-1,
)
resource_ablation_model.fit(
    X_no_resources.loc[masks["train"]], y_train_c,
    categorical_feature=CATEGORICAL_MODEL_FEATURES,
    eval_set=[(X_no_resources.loc[validation_tune_mask], y_tune_c)],
    callbacks=[lgb.early_stopping(60, verbose=False)],
)
ablation_probabilities = resource_ablation_model.predict_proba(X_no_resources.loc[validation_score_mask])
ablation_classification_metrics = classification_metrics(y_score_c, ablation_probabilities)
full_classification_metrics = classification_metrics(y_score_c, selected_specific_model.predict_proba(X_score_c))
KEEP_CLASSIFICATION_RESOURCES = (
    full_classification_metrics["macro_f1"] > ablation_classification_metrics["macro_f1"]
    and full_classification_metrics["log_loss"] <= ablation_classification_metrics["log_loss"]
)
if not KEEP_CLASSIFICATION_RESOURCES:
    print("Resource proxies rejected by validation ablation.")
    classification_artifacts = artifacts_without_resources
    X_all_c = X_no_resources
    selected_specific_base = resource_ablation_model
    selected_specific_model, selected_calibration, _ = calibration_candidates(
        selected_specific_base, X_all_c.loc[calibration_mask], y_cal_c,
        X_all_c.loc[validation_score_mask], y_score_c,
    )
    selected_specific_name = "LightGBM (resource ablation)"
    broad_base = lgb.LGBMClassifier(
        objective="multiclass", n_estimators=700, learning_rate=0.04, num_leaves=40,
        min_child_samples=50, class_weight="balanced", reg_lambda=1.5,
        random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=-1,
    )
    broad_base.fit(
        X_all_c.loc[masks["train"]], y_train_broad, categorical_feature=CATEGORICAL_MODEL_FEATURES,
        eval_set=[(X_all_c.loc[validation_tune_mask], y_tune_broad)],
        callbacks=[lgb.early_stopping(60, verbose=False)],
    )
    selected_broad_model, broad_calibration, _ = calibration_candidates(
        broad_base, X_all_c.loc[calibration_mask], y_cal_broad,
        X_all_c.loc[validation_score_mask], y_score_broad,
    )
else:
    print("Resource proxies retained, with reporting/enforcement-bias limitation documented.")

# Train a genuine hierarchy: broad prediction chooses one broad-specific model.
route_models = {}
for broad_id in np.unique(y_train_broad):
    route_rows = y_train_broad == broad_id
    route_classes = np.unique(y_train_c[route_rows])
    if len(route_classes) == 1:
        route_models[int(broad_id)] = int(route_classes[0])
        continue
    route_model = lgb.LGBMClassifier(
        objective="multiclass", n_estimators=600, learning_rate=0.04, num_leaves=32,
        min_child_samples=35, class_weight="balanced", reg_lambda=1.5,
        random_state=RANDOM_STATE, n_jobs=N_JOBS, verbosity=-1,
    )
    route_model.fit(
        X_all_c.loc[masks["train"]].iloc[np.flatnonzero(route_rows)], y_train_c[route_rows],
        categorical_feature=CATEGORICAL_MODEL_FEATURES,
    )
    route_models[int(broad_id)] = route_model
routed_base = RoutedClassifier(selected_broad_model, route_models, len(specific_encoder.classes_))
routed_model, routed_calibration, routed_calibration_table = calibration_candidates(
    routed_base, X_all_c.loc[calibration_mask], y_cal_c,
    X_all_c.loc[validation_score_mask], y_score_c,
)
routed_probabilities = routed_model.predict_proba(X_all_c.loc[validation_score_mask])
hierarchy_validation_metrics = classification_metrics(y_score_c, routed_probabilities)
hierarchy_size_mb = artifact_size_mb(routed_model, CHECKPOINT_DIR / "classifier_true_hierarchy.pkl")
shared_validation_metrics = classification_metrics(
    y_score_c, selected_specific_model.predict_proba(X_all_c.loc[validation_score_mask])
)
hierarchy_choice = pd.DataFrame([
    {"model_name": selected_specific_name, **shared_validation_metrics, "artifact_size_mb": artifact_size_mb(selected_specific_model, CHECKPOINT_DIR / "classifier_shared_final.pkl")},
    {"model_name": "True routed hierarchy", **hierarchy_validation_metrics, "artifact_size_mb": hierarchy_size_mb},
])
display(routed_calibration_table)
TRUE_HIERARCHY_SELECTED = (
    hierarchy_size_mb <= MAX_DEPLOYMENT_MODEL_MB
    and hierarchy_validation_metrics["macro_f1"] >= shared_validation_metrics["macro_f1"] + MIN_HIERARCHY_MACRO_F1_GAIN
    and hierarchy_validation_metrics["log_loss"] <= shared_validation_metrics["log_loss"]
)
if TRUE_HIERARCHY_SELECTED:
    selected_specific_model = routed_model
    selected_specific_name = "True routed hierarchy"
    selected_calibration = routed_calibration
display(hierarchy_choice.sort_values(["macro_f1", "log_loss"], ascending=[False, True]))
print("True hierarchy selected:", TRUE_HIERARCHY_SELECTED)
selected_specific_validation_metrics = classification_metrics(
    y_score_c, selected_specific_model.predict_proba(X_all_c.loc[validation_score_mask])
)
CLASSIFICATION_VALIDATION_PASSED = selected_specific_validation_metrics["macro_f1"] > float(specific_baseline_table.iloc[0]["macro_f1"])
if CLASSIFICATION_VALIDATION_PASSED:
    print("Classification validation passed its strongest frequency baseline.")
else:
    print("WARNING: classification validation failed its strongest frequency baseline; do not final-test or export this classifier.")


## 13. Final untouched test evaluation — run once

Do not enable this cell while tuning. Set `RUN_FINAL_TEST=True` only after the selected model names, imbalance method, calibration methods, feature ablations, and hyperparameters above are frozen. Forecast and classification have independent validation gates, so one rejected task does not block final evidence for the other. This cell does not feed results back into model choice.


In [ ]:
if not RUN_FINAL_TEST:
    raise RuntimeError(
        "Final test is locked. Review and save validation comparisons, then set RUN_FINAL_TEST=True and run this cell once."
    )
validation_eligibility = {
    "forecast": bool(FORECAST_VALIDATION_PASSED),
    "classification": bool(CLASSIFICATION_VALIDATION_PASSED),
}
if not any(validation_eligibility.values()):
    raise RuntimeError("Final test remains locked because both tasks failed validation.")
print("FINAL TEST UNLOCKED for:", [name for name, passed in validation_eligibility.items() if passed])
print("No model choices may be changed from this point.")

best_baseline_name = "Recursive seven-day rolling average"
forecast_test_rows = pd.DataFrame()
forecast_test_baseline_rows = pd.DataFrame()
forecast_test_metrics = {}
forecast_test_baseline_metrics = {}
forecast_test_inference_ms = None
significance = None
if validation_eligibility["forecast"]:
    forecast_test_rows = recursive_forecast_backtest(
        selected_forecaster, daily, forecast_artifacts, boundaries.test_start, boundaries.test_end,
        horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE,
        prediction_mode=selected_forecast_prediction_mode, model_weight=selected_forecast_model_weight,
    )
    forecast_test_baseline_rows = recursive_forecast_backtest(
        None, daily, forecast_artifacts, boundaries.test_start, boundaries.test_end,
        horizon=FORECAST_HORIZON, stride=FORECAST_ORIGIN_STRIDE, model_weight=0.0,
    )
    forecast_keys = ["origin", "date", "city", "horizon"]
    assert forecast_test_rows[forecast_keys].equals(forecast_test_baseline_rows[forecast_keys])
    forecast_test_prediction = forecast_test_rows["predicted"].to_numpy()
    forecast_test_actual = forecast_test_rows["actual"].to_numpy()
    forecast_test_metrics = regression_metrics(forecast_test_actual, forecast_test_prediction)
    forecast_test_baseline_metrics = {best_baseline_name: regression_metrics(
        forecast_test_baseline_rows["actual"], forecast_test_baseline_rows["predicted"]
    )}
    _, forecast_test_inference_ms = predict_timed(
        selected_forecaster, forecast_frame.loc[forecast_masks["test"], forecast_artifacts["feature_columns"]]
    )
    significance = paired_absolute_error_bootstrap(
        forecast_test_actual, forecast_test_prediction, forecast_test_baseline_rows["predicted"].to_numpy(), RANDOM_STATE,
        groups=forecast_test_rows["origin"].astype(str) + "|" + forecast_test_rows["city"].astype(str),
    )
else:
    print("Forecast final test skipped because it did not beat validation baseline.")

specific_test_probabilities = None
broad_test_probabilities = None
specific_test_metrics = {}
broad_test_metrics = {}
specific_test_baselines = {}
specific_test_baseline_metrics = {}
specific_test_inference_ms = None
broad_test_inference_ms = None
classification_significance = None
best_classification_baseline = specific_baseline_table.iloc[0]["model_name"]
if validation_eligibility["classification"]:
    X_test_c = X_all_c.loc[masks["test"]]
    y_test_specific = specific_labels[masks["test"].to_numpy()]
    y_test_broad = broad_labels[masks["test"].to_numpy()]
    specific_test_probabilities, specific_test_inference_ms = predict_timed(selected_specific_model, X_test_c, probabilities=True)
    broad_test_probabilities, broad_test_inference_ms = predict_timed(selected_broad_model, X_test_c, probabilities=True)
    specific_test_metrics = classification_metrics(y_test_specific, specific_test_probabilities)
    broad_test_metrics = classification_metrics(y_test_broad, broad_test_probabilities)
    specific_test_baselines = classification_baselines(
        train_context, specific_labels[masks["train"].to_numpy()],
        df.loc[masks["test"], ["city", "location_type", "hour"]],
        len(specific_encoder.classes_),
    )
    specific_test_baseline_metrics = {
        name: classification_metrics(y_test_specific, probabilities)
        for name, probabilities in specific_test_baselines.items()
    }
    classification_significance = paired_accuracy_bootstrap(
        y_test_specific, specific_test_probabilities.argmax(axis=1),
        specific_test_baselines[best_classification_baseline].argmax(axis=1), RANDOM_STATE,
    )
else:
    print("Classification final test skipped because it did not beat its validation baseline.")
qualification = {
    "forecast_beats_validation_selected_baseline": (
        bool(forecast_test_metrics["mae"] < forecast_test_baseline_metrics[best_baseline_name]["mae"])
        if validation_eligibility["forecast"] else None
    ),
    "forecast_improvement_statistically_supported": (
        bool(significance["ci_95_upper"] < 0) if significance is not None else None
    ),
    "specific_classifier_beats_validation_selected_baseline": (
        bool(specific_test_metrics["macro_f1"] > specific_test_baseline_metrics[best_classification_baseline]["macro_f1"])
        if validation_eligibility["classification"] else None
    ),
    "classification_accuracy_gain_statistically_supported": (
        bool(classification_significance["ci_95_lower"] > 0) if classification_significance is not None else None
    ),
}

final_results = {
    "task_status": {
        name: "evaluated" if passed else "skipped_validation_gate"
        for name, passed in validation_eligibility.items()
    },
    "forecast": forecast_test_metrics,
    "broad_classification": broad_test_metrics,
    "specific_classification": specific_test_metrics,
    "forecast_vs_best_baseline_bootstrap": significance,
    "classification_vs_best_baseline_bootstrap": classification_significance,
    "forecast_baselines": forecast_test_baseline_metrics,
    "forecast_protocol": {"horizon_days": FORECAST_HORIZON, "origin_stride_days": FORECAST_ORIGIN_STRIDE, "model_weight": selected_forecast_model_weight, "prediction_mode": selected_forecast_prediction_mode},
    "specific_classification_baselines": specific_test_baseline_metrics,
    "qualification": qualification,
    "inference_ms_per_row": {
        "forecast": forecast_test_inference_ms,
        "broad_classification": broad_test_inference_ms,
        "specific_classification": specific_test_inference_ms,
    },
}
_ = (REPORTS_DIR / "final_test_results.json").write_text(json.dumps(final_results, indent=2))
print(json.dumps(final_results, indent=2))


## 14. Final diagnostics and poster graphs

These plots use only locked final-test predictions from tasks that passed validation. A qualifying forecast produces a complete 30-day trajectory and subgroup errors; a qualifying classifier produces confusion matrices, class metrics, reliability, and top-k accuracy.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay

# Poster-ready boosting progress (validation only; boosting rounds are not epochs).
forecast_history = forecast_candidates.get(selected_forecast_candidate_name, {}).get("evaluation_history", {})
if forecast_history:
    dataset_name = next(iter(forecast_history))
    metric_name = next(iter(forecast_history[dataset_name]))
    plt.figure(figsize=(7, 4.5)); plt.plot(forecast_history[dataset_name][metric_name], linewidth=2)
    plt.title("Volume Model Validation Progress"); plt.xlabel("Boosting round"); plt.ylabel(metric_name)
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "volume_validation_progress.png", dpi=180); plt.show()
classification_history = imbalance_histories.get(selected_imbalance, {})
if classification_history:
    dataset_name = next(iter(classification_history))
    metric_name = next(iter(classification_history[dataset_name]))
    plt.figure(figsize=(7, 4.5)); plt.plot(classification_history[dataset_name][metric_name], linewidth=2)
    plt.title("Specific Model Validation Progress"); plt.xlabel("Boosting round"); plt.ylabel(metric_name)
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "specific_validation_progress.png", dpi=180); plt.show()

if not forecast_test_rows.empty:
    test_forecast_rows = forecast_test_rows.rename(columns={"actual": "crime_count"}).copy()
    test_forecast_rows["absolute_error"] = abs(test_forecast_rows["predicted"] - test_forecast_rows["crime_count"])

    plot_city = test_forecast_rows.groupby("city")["crime_count"].mean().sort_values(ascending=False).index[0]
    plot_origin = test_forecast_rows["origin"].min()
    plot_data = test_forecast_rows[(test_forecast_rows["city"] == plot_city) & (test_forecast_rows["origin"] == plot_origin)]
    plt.figure(figsize=(11, 5))
    plt.plot(plot_data["date"], plot_data["crime_count"], label="Actual", linewidth=2)
    plt.plot(plot_data["date"], plot_data["predicted"], label="Predicted", linewidth=2)
    plt.title(f"Untouched 30-Day Test Forecast: {plot_city} ({plot_origin.date()})")
    plt.ylabel("Daily reported incidents"); plt.xlabel("Date"); plt.legend(); plt.tight_layout()
    plt.savefig(PLOTS_DIR / "forecast_actual_vs_predicted_test.png", dpi=180); plt.show()

    forecast_by_city = test_forecast_rows.groupby("city").apply(
        lambda rows: pd.Series(regression_metrics(rows["crime_count"], rows["predicted"])), include_groups=False
    ).reset_index()
    forecast_by_month = test_forecast_rows.assign(month=lambda d: d.date.dt.to_period("M").astype(str)).groupby("month").apply(
        lambda rows: pd.Series(regression_metrics(rows["crime_count"], rows["predicted"])), include_groups=False
    ).reset_index()
    forecast_by_weekday = test_forecast_rows.assign(weekday=lambda d: d.date.dt.day_name()).groupby("weekday").apply(
        lambda rows: pd.Series(regression_metrics(rows["crime_count"], rows["predicted"])), include_groups=False
    ).reset_index()
    forecast_by_city.to_csv(REPORTS_DIR / "forecast_metrics_by_city.csv", index=False)
    forecast_by_month.to_csv(REPORTS_DIR / "forecast_metrics_by_month.csv", index=False)
    forecast_by_weekday.to_csv(REPORTS_DIR / "forecast_metrics_by_weekday.csv", index=False)

if specific_test_probabilities is not None:
    specific_predictions = specific_test_probabilities.argmax(axis=1)
    specific_per_class = per_class_metrics(y_test_specific, specific_predictions, specific_encoder.classes_.tolist())
    specific_per_class.to_csv(REPORTS_DIR / "specific_metrics_by_class.csv", index=False)

    test_context = df.loc[masks["test"], ["date", "city", "location_type"]].reset_index(drop=True)
    def classification_metrics_by_group(group_column):
        rows = []
        for group_value, positions in test_context.groupby(group_column).groups.items():
            positions = np.asarray(list(positions), dtype=int)
            if len(positions) < 20:
                continue
            rows.append({group_column: group_value, "rows": len(positions),
                         **classification_metrics(y_test_specific[positions], specific_test_probabilities[positions])})
        return pd.DataFrame(rows)
    classification_metrics_by_group("city").to_csv(REPORTS_DIR / "classification_metrics_by_city.csv", index=False)
    test_context["year"] = test_context["date"].dt.year
    classification_metrics_by_group("year").to_csv(REPORTS_DIR / "classification_metrics_by_year.csv", index=False)
    classification_metrics_by_group("location_type").to_csv(REPORTS_DIR / "classification_metrics_by_location_type.csv", index=False)
    ConfusionMatrixDisplay.from_predictions(
        y_test_specific, specific_predictions, display_labels=specific_encoder.classes_, normalize="true",
        xticks_rotation=45, cmap="Blues", values_format=".0%", figsize=(13, 10),
    )
    plt.title("Specific Offense Confusion Matrix: Untouched Test")
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "specific_confusion_matrix_test.png", dpi=180); plt.show()

    broad_predictions = broad_test_probabilities.argmax(axis=1)
    ConfusionMatrixDisplay.from_predictions(
        y_test_broad, broad_predictions, display_labels=broad_encoder.classes_, normalize="true",
        cmap="Blues", values_format=".0%",
    )
    plt.title("Broad Category Confusion Matrix: Untouched Test")
    plt.tight_layout(); plt.savefig(PLOTS_DIR / "broad_confusion_matrix_test.png", dpi=180); plt.show()

    topk_names = ["Top-1", "Top-3"]
    topk_values = [specific_test_metrics["accuracy"], specific_test_metrics["top_3_accuracy"]]
    plt.figure(figsize=(6, 4.5)); bars = plt.bar(topk_names, np.array(topk_values) * 100, color=["#4C78A8", "#59A14F"])
    plt.ylabel("Accuracy (%)"); plt.title("Specific Offense Top-K Accuracy: Untouched Test")
    plt.bar_label(bars, fmt="%.1f%%"); plt.ylim(0, 100); plt.tight_layout()
    plt.savefig(PLOTS_DIR / "specific_topk_accuracy_test.png", dpi=180); plt.show()

    confidence = specific_test_probabilities.max(axis=1)
    correct = (specific_predictions == y_test_specific).astype(int)
    prob_true, prob_pred = calibration_curve(correct, confidence, n_bins=10, strategy="quantile")
    plt.figure(figsize=(6, 5)); plt.plot(prob_pred, prob_true, marker="o", label="Selected model")
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    plt.xlabel("Mean confidence"); plt.ylabel("Observed accuracy"); plt.title("Reliability: Untouched Test")
    plt.legend(); plt.tight_layout(); plt.savefig(PLOTS_DIR / "specific_reliability_test.png", dpi=180); plt.show()



## 15. Model comparison table and artifact export

The app-compatible filenames are preserved. Per-city experiments remain diagnostic and are not exported in this compact deployment. `feature_artifacts.pkl` is the shared training/inference contract, and `app_data_bundle.pkl` replaces the incident-level CSV at runtime. The manifest records schemas, metrics, package versions, artifact sizes, and Git commit.


In [ ]:
if not RUN_FINAL_TEST or not (REPORTS_DIR / "final_test_results.json").exists():
    raise RuntimeError("Do not export final artifacts before the untouched final-test evaluation succeeds.")
if not qualification["forecast_beats_validation_selected_baseline"]:
    raise RuntimeError("Forecast failed its untouched-test baseline gate. Record the result, but do not export it as improved.")
if not qualification["specific_classifier_beats_validation_selected_baseline"]:
    raise RuntimeError("Classifier failed its untouched-test baseline gate. Record the result, but do not export it as improved.")

comparison_rows = []
for _, baseline in forecast_baseline_table.iterrows():
    validation_metrics = {key: baseline[key] for key in ["mae", "rmse", "wape", "smape", "poisson_deviance", "mean_bias"]}
    comparison_rows.append(comparison_row(
        "Volume forecast", baseline["model_name"], "Historical baseline", periods,
        validation_metrics, forecast_test_baseline_metrics[baseline["model_name"]],
        decision="Baseline", reason="Reference baseline",
    ))
for name, result in forecast_candidates.items():
    selected_candidate = KEEP_FORECAST_RESOURCES and name == selected_forecast_candidate_name
    candidate_validation_metrics = selected_forecast_validation_metrics if selected_candidate else result["metrics"]
    comparison_rows.append(comparison_row(
        "Volume forecast", name, "Recursive 30-day forecast v2", periods, candidate_validation_metrics,
        forecast_test_metrics if selected_candidate else None,
        hyperparameters=result["model"].get_params(), training_time_seconds=result["training_seconds"],
        inference_ms_per_row=result["inference_ms"], artifact_size_mb=result["artifact_size_mb"],
        decision="Selected" if selected_candidate else "Rejected",
        reason=f"Best recursive validation MAE; model weight={selected_forecast_model_weight:.2f}" if selected_candidate else "Recursive validation ranking or resource ablation",
    ))
if not KEEP_FORECAST_RESOURCES:
    comparison_rows.append(comparison_row(
        "Volume forecast", selected_forecast_name, "Recursive 30-day forecast v2 without resource proxies",
        periods, ablation_metrics, forecast_test_metrics, decision="Selected",
        reason=f"Resource ablation improved recursive validation MAE; model weight={selected_forecast_model_weight:.2f}",
    ))
for name, result in classifier_candidates.items():
    metrics = classification_metrics(y_score_c, result["probabilities"])
    comparison_rows.append(comparison_row(
        "Specific classification", name, "Shared classification v2", periods, metrics,
        specific_test_metrics if KEEP_CLASSIFICATION_RESOURCES and not TRUE_HIERARCHY_SELECTED and name == selected_specific_candidate_name else None,
        imbalance_method=result["imbalance_method"], calibration_method=selected_calibration,
        training_time_seconds=result["training_seconds"], inference_ms_per_row=result["inference_ms"],
        artifact_size_mb=result["artifact_size_mb"], decision="Selected" if KEEP_CLASSIFICATION_RESOURCES and not TRUE_HIERARCHY_SELECTED and name == selected_specific_candidate_name else "Rejected",
        reason="Validation selection priority" if KEEP_CLASSIFICATION_RESOURCES and not TRUE_HIERARCHY_SELECTED and name == selected_specific_candidate_name else "Validation ranking, resource ablation, or hierarchy comparison",
    ))
for _, baseline in specific_baseline_table.iterrows():
    validation_metrics = {key: baseline[key] for key in ["accuracy", "top_3_accuracy", "macro_f1", "weighted_f1", "balanced_accuracy", "log_loss", "brier_multiclass", "ece"]}
    comparison_rows.append(comparison_row(
        "Specific classification", baseline["model_name"], "Historical frequency baseline", periods,
        validation_metrics, specific_test_baseline_metrics[baseline["model_name"]],
        decision="Baseline", reason="Reference baseline",
    ))
if not KEEP_CLASSIFICATION_RESOURCES:
    comparison_rows.append(comparison_row(
        "Specific classification", selected_specific_name, "Shared classification v2 without resource proxies",
        periods, ablation_classification_metrics, specific_test_metrics, imbalance_method="class_weight",
        calibration_method=selected_calibration, decision="Selected" if not TRUE_HIERARCHY_SELECTED else "Rejected",
        reason="Resource ablation improved validation macro-F1/log loss",
    ))
broad_validation_metrics = classification_metrics(
    y_score_broad, selected_broad_model.predict_proba(X_all_c.loc[validation_score_mask])
)
comparison_rows.append(comparison_row(
    "Broad classification", "LightGBM broad classifier", "Shared classification v2",
    periods, broad_validation_metrics, broad_test_metrics, imbalance_method="class_weight",
    calibration_method=broad_calibration, inference_ms_per_row=broad_test_inference_ms,
    artifact_size_mb=artifact_size_mb(selected_broad_model, CHECKPOINT_DIR / "classifier_broad_final.pkl"),
    decision="Selected", reason="Required broad probability layer selected on validation calibration metrics",
))
comparison_rows.append(comparison_row(
    "Specific classification", "True routed hierarchy", "Broad-routed classification v2",
    periods, hierarchy_validation_metrics, specific_test_metrics if TRUE_HIERARCHY_SELECTED else None,
    imbalance_method="class_weight", calibration_method=routed_calibration,
    artifact_size_mb=hierarchy_size_mb, decision="Selected" if TRUE_HIERARCHY_SELECTED else "Rejected",
    reason="Validation hierarchy comparison",
))
comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
display(comparison_table)

validation_residuals = selected_forecast_validation_rows["actual"].to_numpy() - selected_forecast_validation_rows["predicted"].to_numpy()
forecast_artifacts["residual_quantiles"] = {
    "lower": float(np.quantile(validation_residuals, 0.05)),
    "upper": float(np.quantile(validation_residuals, 0.95)),
}
feature_artifacts = {
    "schema_version": SCHEMA_VERSION,
    "model_version": MODEL_VERSION,
    "forecast": forecast_artifacts,
    "classification": classification_artifacts,
    "rare_class_mapping": rare_mapping,
    "temporal_boundaries": boundaries.to_dict(),
}

# Compact dashboard summaries replace the 298 MB incident-level CSV at
# Streamlit runtime while preserving every existing chart and selector.
app_daily = daily.copy()
resource_columns = [column for column in RESOURCE_FEATURES if column in app_daily]
if resource_columns:
    app_daily[resource_columns] = app_daily.groupby("city", observed=True)[resource_columns].ffill().fillna(0)
officer_trends = df.assign(month_start=df["date"].dt.to_period("M").dt.to_timestamp())[
    ["month_start", "city", "population", "total_officers", "male_officer",
     "female_officer", "officers_per_1000_people"]
].drop_duplicates()
crime_distribution = (
    df.assign(year=df["date"].dt.year)
    .groupby(["city", "year", "offense_category_name"], observed=True)
    .size().rename("count").reset_index()
)
app_data_bundle = {
    "schema_version": SCHEMA_VERSION,
    "model_version": MODEL_VERSION,
    "daily_city": app_daily,
    "officer_trends": officer_trends,
    "crime_distribution": crime_distribution,
    "location_areas": sorted(df["location_area"].astype(str).unique().tolist()),
    "years": sorted(df["date"].dt.year.unique().astype(int).tolist()),
}

artifacts = {
    "crime_forecaster.pkl": selected_forecaster,
    "forecast_features.pkl": forecast_artifacts["feature_columns"],
    "crime_classifier_l1_violent_property.pkl": selected_broad_model,
    "crime_classifier_l2_specific.pkl": selected_specific_model,
    "label_encoder_l1.pkl": broad_encoder,
    "label_encoder_l2.pkl": specific_encoder,
    "advanced_features.pkl": classification_artifacts["feature_columns"],
    "feature_artifacts.pkl": feature_artifacts,
    "app_data_bundle.pkl": app_data_bundle,
}

artifact_paths = []
for filename, value in artifacts.items():
    path = ARTIFACTS_DIR / filename
    joblib.dump(value, path, compress=3)
    artifact_paths.append(path)
    print(f"Saved {filename}: {path.stat().st_size / 1024**2:.2f} MB")

# Export generated evidence alongside the deployment files.
for report_name in ["model_comparison.csv", "final_test_results.json", "temporal_boundaries.json"]:
    source = REPORTS_DIR / report_name
    destination = ARTIFACTS_DIR / report_name
    shutil.copy2(source, destination)
    artifact_paths.append(destination)
generated_model_card = ARTIFACTS_DIR / "MODEL_CARD.generated.md"
generated_model_card.write_text(
    "# ProjeCT360 Model Card (Generated)\n\n"
    + f"Model version: {MODEL_VERSION}\n\n"
    + f"Dataset: {df['date'].min().date()} through {df['date'].max().date()}\n\n"
    + f"Selected forecast model: {selected_forecast_name}\n\n"
    + f"Forecast protocol: recursive {FORECAST_HORIZON}-day trajectories; model weight {selected_forecast_model_weight:.2f}; mode {selected_forecast_prediction_mode}\n\n"
    + f"Selected broad classifier: LightGBM ({broad_calibration})\n\n"
    + f"Selected specific classifier: {selected_specific_name} ({selected_calibration})\n\n"
    + f"Evaluation periods: {periods}\n\n"
    + "## Untouched final-test metrics\n\n```json\n" + json.dumps(final_results, indent=2) + "\n```\n\n"
    + "## Intended use\nEducational aggregate analysis of historically reported Connecticut incidents.\n\n"
    + "## Prohibited use\nNo person-level, address-level, enforcement, patrol, sentencing, or resource-allocation decisions.\n\n"
    + "## Targets\nDaily city-level reported-incident counts and broad/specific reported-offense categories.\n\n"
    + "## Evaluation\nChronological training, recursive 30-day validation origins, internal classification validation slices, and one locked final test. Calibration, blend weight, feature ablations, and all model choices use validation only.\n\n"
    + "## Limitations\nResults can reflect reporting access, enforcement patterns, missing data, geographic coverage, class imbalance, and future distribution shift. Staffing and population context use prior completed years only and are retained only after ablation. Probabilities are uncertain estimates, not statements that an event will occur.\n"
)
artifact_paths.append(generated_model_card)

manifest = write_manifest(
    ARTIFACTS_DIR / "model_manifest.json", MODEL_VERSION,
    {"start": str(df["date"].min().date()), "end": str(df["date"].max().date())},
    {"forecast": selected_forecast_name, "broad_classifier": "LightGBM", "specific_classifier": selected_specific_name},
    {"forecast": forecast_artifacts["feature_columns"], "classification": classification_artifacts["feature_columns"]},
    final_results, artifact_paths, PROJECT_ROOT,
)
artifact_paths.append(ARTIFACTS_DIR / "model_manifest.json")
print("Manifest model version:", manifest["model_version"])


## 16. Optional Hugging Face upload

For a private repository, create a Hugging Face **read/write** token only for this Colab session. Streamlit should continue using a separate read-only token. The upload includes only selected deployment artifacts and reports, not redundant per-city models or Git history.


In [ ]:
if UPLOAD_TO_HUGGING_FACE:
    from getpass import getpass
    from huggingface_hub import HfApi

    hf_token = os.getenv("HF_TOKEN") or getpass("Hugging Face write token: ")
    api = HfApi(token=hf_token)
    api.create_repo(HF_REPO_ID, repo_type=HF_REPO_TYPE, private=True, exist_ok=True)
    obsolete_remote_files = [
        "per_city_models.pkl", "per_city_forecasters.pkl", "per_city_forecast_features.pkl",
        "per_city_accuracies.pkl", "performance_summary.pkl", "classifier_features.pkl",
        "crime_classifier.pkl", "label_encoder.pkl", "big_cities.pkl",
        "city_target_enc.pkl", "class_thresholds.pkl",
    ]
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        folder_path=ARTIFACTS_DIR,
        path_in_repo="",
        delete_patterns=obsolete_remote_files,
        commit_message=f"Upload ProjeCT360 model artifacts {MODEL_VERSION}",
    )
    print("Uploaded selected artifacts to", HF_REPO_ID)
else:
    print("UPLOAD_TO_HUGGING_FACE=False; artifacts remain in OUTPUT_DIR.")


## 17. Final compatibility verification

This reloads every required artifact, checks app-facing output contracts, rejects target columns, and records a verification report. It does not modify `app.py`.


In [ ]:
required_files = [
    "crime_forecaster.pkl", "forecast_features.pkl",
    "crime_classifier_l1_violent_property.pkl", "crime_classifier_l2_specific.pkl",
    "label_encoder_l1.pkl", "label_encoder_l2.pkl", "advanced_features.pkl",
    "feature_artifacts.pkl", "app_data_bundle.pkl", "model_manifest.json",
]
missing = [name for name in required_files if not (ARTIFACTS_DIR / name).exists()]
assert not missing, f"Missing final artifacts: {missing}"

reloaded_features = joblib.load(ARTIFACTS_DIR / "feature_artifacts.pkl")
assert reloaded_features["schema_version"] == SCHEMA_VERSION
reloaded_forecast_features = joblib.load(ARTIFACTS_DIR / "forecast_features.pkl")
assert_safe_features(reloaded_forecast_features)
reloaded_forecast_contract = reloaded_features["forecast"]
assert list(reloaded_forecast_features) == list(reloaded_forecast_contract["feature_columns"])
assert reloaded_forecast_contract["prediction_mode"] in FORECAST_PREDICTION_MODES
assert 0.0 <= float(reloaded_forecast_contract["model_weight"]) <= 1.0
assert int(reloaded_forecast_contract["forecast_horizon"]) == FORECAST_HORIZON
assert_safe_features(joblib.load(ARTIFACTS_DIR / "advanced_features.pkl"))
reloaded_bundle = joblib.load(ARTIFACTS_DIR / "app_data_bundle.pkl")
assert {"daily_city", "officer_trends", "crime_distribution", "location_areas"}.issubset(reloaded_bundle)
deployment_sizes_mb = {
    name: (ARTIFACTS_DIR / name).stat().st_size / 1024**2
    for name in ["crime_forecaster.pkl", "crime_classifier_l1_violent_property.pkl",
                 "crime_classifier_l2_specific.pkl"]
}
assert deployment_sizes_mb["crime_classifier_l2_specific.pkl"] <= MAX_DEPLOYMENT_MODEL_MB, (
    "Selected classifier exceeds the configured Streamlit deployment budget."
)

forecast_contract_columns = {"Date", "Predicted Count", "is_holiday"}
risk_contract_fields = {
    "broad_label", "broad_probability", "broad_probs", "broad_model_classes",
    "specific_label", "specific_probability", "specific_probs", "specific_model_classes", "model_source",
}
verification = {
    "status": "passed",
    "model_version": MODEL_VERSION,
    "required_files": required_files,
    "deployment_sizes_mb": deployment_sizes_mb,
    "forecast_output_contract": sorted(forecast_contract_columns),
    "forecast_strategy": {
        "prediction_mode": reloaded_forecast_contract["prediction_mode"],
        "model_weight": float(reloaded_forecast_contract["model_weight"]),
        "forecast_horizon": int(reloaded_forecast_contract["forecast_horizon"]),
    },
    "risk_output_contract": sorted(risk_contract_fields),
    "frontend_modified_by_notebook": False,
}
(REPORTS_DIR / "final_verification.json").write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))


## 18. Responsible-use checklist

- Intended for education and aggregate historical-pattern exploration.
- Not for person-level prediction, address-level targeting, enforcement allocation, or claims that a place/person is dangerous.
- Predictions concern **reported incidents**, which reflect reporting access, missingness, enforcement patterns, and uneven geographic coverage.
- Probability does not mean certainty; confidence and calibration are limited by distribution shift and class imbalance.
- Staffing and crime-rate features are included only if the validation ablation supports them, and their proxy risks remain documented.
- The final model card and reports must quote only the generated untouched-test metrics from this run.
